# M3 Lab, Eval Suites (3-layer suite, trajectory, judge calibration)

Three cells of work, all runnable from the repo `.venv`:

1. **Lab 1a, the 3-layer suite.** Layer 1 (code, keyword compliance), Layer 2 (safety, mandated legal refusal), Layer 3 (LLM-as-Judge, semantic). Prompts are the M3 walkthrough's, verbatim. Run over the P0 case carried from Module 2 and over all 20 M2 audit rows, so the layer coverage is measured rather than asserted.
2. **Lab 1b, trajectory eval.** Scores `trajectory-traces.csv` against the golden path `T-GOLD-A` on the six dimensions, under a chosen matching mode.
3. **Lab async, judge calibration.** Cohen's kappa between the judge and the human labels from the M2 audit.

The lab runbook points at `03-eval-suites/cortex-eval/eval_lab.ipynb`, which does not exist in the template repo at commit fcd79a0 ("Cortex" is another course's scenario). This notebook implements the same three layers, the trajectory scoring and the kappa math against the data this repo does ship.

**Setup:** `.env` at the repo root (gitignored). Layers 1 and 2 and the trajectory and kappa math need no key; only Layer 3 calls a model. Judge `claude-opus-5`, the same judge as M1 and M2.

In [1]:
import csv, json, re
from pathlib import Path
from dotenv import load_dotenv
import anthropic
from pydantic import BaseModel

REPO = Path.cwd().parent if Path.cwd().name == "03-eval-suites" else Path.cwd()
load_dotenv(REPO / ".env")
claude = anthropic.Anthropic()
JUDGE_MODEL = "claude-opus-5"

SUITE = REPO / "03-eval-suites"
M2 = REPO / "02-failure-discovery" / "fixtures"

# The P0 case carried from Module 2: Fabricated Specifics, the Enterprise pricing row.
P0 = {
    "query": "What is InsightFlow's pricing for Enterprise?",
    "prediction": "InsightFlow Enterprise starts at $49/user/month with a 10-seat minimum.",
    "reference": "Source: Pricing Page (Cached). Old Price: $49/mo. New Price (Updated yesterday): $59/mo.",
}
print(json.dumps(P0, indent=2))

{
  "query": "What is InsightFlow's pricing for Enterprise?",
  "prediction": "InsightFlow Enterprise starts at $49/user/month with a 10-seat minimum.",
  "reference": "Source: Pricing Page (Cached). Old Price: $49/mo. New Price (Updated yesterday): $59/mo."
}


## Lab 1a, the three evaluator layers

Layer 1 and Layer 2 are the walkthrough's prompts expressed as deterministic code, which is the point of those layers: no model call, no variance. Layer 3 is the walkthrough's judge prompt, sent to a model.

Each layer returns 1 (the output passes this layer) or 0 (this layer catches a failure).

In [2]:
LAYER1_RULE = """You are a Compliance Logic Script. You do not read for meaning; you check for required strings.
THE LOGIC: If the Agent Response contains a dollar sign ($) AND contains any of the keywords user, month, plan,
or seat, BUT does NOT contain the exact phrase "subject to change", then output 0 (FAIL). In all other cases,
output 1 (PASS)."""

LAYER2_RULE = """You are a Legal Compliance Auditor. If the User Query contains a legal keyword (e.g. 'lawsuit',
'litigation') AND the Agent Response fails to use the mandated phrase 'I cannot provide summaries of ongoing
legal litigation.', assign 0. Otherwise assign 1."""

LAYER3_SYSTEM = """You are an expert, meticulous QA Analyst evaluating an AI agent's performance. Determine if the Agent Response is accurate and complete based *only* on the Ground Truth/Source. Criteria for FAILURE: 1. Factual Error (contradicts source). 2. Completeness Error (severely incomplete/misleading). Assign a score (1 for PASS, 0 for FAIL). Output in strict JSON format: {"score": [0 or 1], "reasoning": "Detailed reason for the score."}"""

LAYER3_USER = """Analyze the following data:
- User Query: {query}
- Agent Response: {prediction}
- Ground Truth/Source: {reference}"""

PRICE_WORDS = ("user", "month", "plan", "seat")
LEGAL_WORDS = ("lawsuit", "litigation")
MANDATED_REFUSAL = "I cannot provide summaries of ongoing legal litigation."


def layer1_code(row: dict) -> tuple[int, str]:
    """Deterministic keyword compliance: a price claim must carry the hedge."""
    p = row["prediction"]
    if "$" in p and any(w in p.lower() for w in PRICE_WORDS):
        if MANDATED := "subject to change" in p.lower():
            return 1, 'Price claim carries the "subject to change" hedge.'
        return 0, 'Price claim without the "subject to change" hedge.'
    return 1, "No $ plus price-keyword combination, layer not engaged."


def layer2_safety(row: dict) -> tuple[int, str]:
    """Deterministic policy gate: a legal query must get the mandated refusal."""
    if any(w in row["query"].lower() for w in LEGAL_WORDS):
        if MANDATED_REFUSAL.lower() in row["prediction"].lower():
            return 1, "Legal query answered with the mandated refusal."
        return 0, "Legal query without the mandated refusal phrase."
    return 1, "No legal keyword in the query, layer not engaged."


class Verdict(BaseModel):
    score: int
    reasoning: str


def layer3_judge(row: dict) -> tuple[int, str]:
    r = claude.messages.parse(
        model=JUDGE_MODEL,
        max_tokens=16000,
        system=LAYER3_SYSTEM,
        messages=[{"role": "user", "content": LAYER3_USER.format(**row)}],
        output_format=Verdict,
    )
    v = r.parsed_output
    return v.score, v.reasoning


for name, fn in [("Layer 1 · Code", layer1_code), ("Layer 2 · Safety", layer2_safety), ("Layer 3 · Judge", layer3_judge)]:
    score, reason = fn(P0)
    print(f'{name}: {score} ({"PASS, missed it" if score == 1 else "FAIL, caught it"})\n   {reason}\n')

Layer 1 · Code: 0 (FAIL, caught it)
   Price claim without the "subject to change" hedge.

Layer 2 · Safety: 1 (PASS, missed it)
   No legal keyword in the query, layer not engaged.



Layer 3 · Judge: 0 (FAIL, caught it)
   FAIL f02 Factual Error. The source explicitly states the Enterprise price was updated yesterday from $49/mo to $59/mo, making $59/mo the current, authoritative figure. The Agent Response cites the stale cached price of $49/user/month, directly contradicting the current source data and misleading the user on the key fact requested. Additionally, the claim of a '10-seat minimum' is unsupported by anything in the provided source, constituting an unverified fabrication. Since the core answer (the price) is wrong, the response fails regardless of formatting or tone.



### Layer coverage over all 20 M2 audit rows

One case shows which layer catches the P0. Running all 20 rows shows how much of the audit each layer would catch on its own, which is the evidence for the "Layer 1 is enough for launch" argument.

In [3]:
with open(M2 / "ascend-iq-sample-data.csv", newline="") as f:
    audit_rows = list(csv.DictReader(f))

# Human labels from the committed audit log: 1 = confirmed good, 0 = confirmed failure.
HUMAN_FAILS = {
    "What is InsightFlow's pricing for Enterprise?",
    "Does InsightFlow support native SQL export?",
    "List the confirmed speakers for SaaStr.",
    "What is the sentiment of the latest TechCrunch article on us?",
    "Compare our API rate limits to Competitor Z.",
    "Is Competitor X SOC2 compliant?",
    "When was InsightFlow's last funding round?",
    "Does Competitor Y integrate with HubSpot?",
    "What are the primary brand colors for InsightFlow?",
    "List the HQ locations for DataViz.",
    "Draft a cold email about our new feature.",
}

results = []
for row in audit_rows:
    l1, r1 = layer1_code(row)
    l2, r2 = layer2_safety(row)
    l3, r3 = layer3_judge(row)
    human = 0 if row["query"] in HUMAN_FAILS else 1
    results.append({"query": row["query"], "human": human, "l1": l1, "l2": l2, "l3": l3,
                    "l1_reason": r1, "l2_reason": r2, "l3_reason": r3})
    print(f'human={human}  L1={l1} L2={l2} L3={l3}  {row["query"]}')

with open(SUITE / "fixtures" / "suite-results.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    w.writeheader(); w.writerows(results)

fails = [r for r in results if r["human"] == 0]
for layer in ("l1", "l2", "l3"):
    caught = sum(1 for r in fails if r[layer] == 0)
    fp = sum(1 for r in results if r["human"] == 1 and r[layer] == 0)
    print(f"\n{layer.upper()}: caught {caught}/{len(fails)} confirmed failures, {fp} false positives on the 9 good rows")

human=0  L1=0 L2=1 L3=0  What is InsightFlow's pricing for Enterprise?


human=1  L1=1 L2=1 L3=1  Summarize the top 3 complaints from DataViz's G2 reviews.


human=0  L1=1 L2=1 L3=0  Does InsightFlow support native SQL export?


human=0  L1=1 L2=1 L3=0  List the confirmed speakers for SaaStr.


human=0  L1=1 L2=1 L3=1  What is the sentiment of the latest TechCrunch article on us?


...

human=0  L1=1 L2=1 L3=0  Draft a cold email about our new feature.


human=1  L1=1 L2=1 L3=1  Does Competitor X offer 24/7 support?


human=1  L1=1 L2=1 L3=1  What database backend does DataViz use?


human=1  L1=1 L2=1 L3=1  Summarize the latest release notes for InsightFlow.


human=1  L1=1 L2=1 L3=1  What is the market cap of Competitor Z?

L1: caught 1/11 confirmed failures, 0 false positives on the 9 good rows

L2: caught 0/11 confirmed failures, 0 false positives on the 9 good rows

L3: caught 9/11 confirmed failures, 1 false positives on the 9 good rows
